In [1]:
1+1

2

In [22]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("XAI_API_KEY"),
    base_url="https://api.x.ai/v1"
)

In [23]:
response = client.chat.completions.create(
    model="grok-4.7",
    messages=[
        {
            "role": "user",
            "content": "What is 125 multiplied by 24?"
        }
    ]
)

print(response.choices[0].message.content)

''''
We send a prompt to Grok.
Grok generates a text response.
No custom Python function is registered or executed by our application.
The model might calculate correctly, but the result is not guaranteed to be exact.'''

**3000**

125 × 24 = 3000.


"'\nWe send a prompt to Grok.\nGrok generates a text response.\nNo custom Python function is registered or executed by our application.\nThe model might calculate correctly, but the result is not guaranteed to be exact."

#tools

In [24]:
def multiply(a, b):
    print(f"Python is executing: {a} * {b}")
    return a * b

In [27]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two numbers accurately.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "number",
                        "description": "First number"
                    },
                    "b": {
                        "type": "number",
                        "description": "Second number"
                    }
                },
                "required": ["a", "b"]
            }
        }
    }
]

In [28]:
messages = [
    {
        "role": "user",
        "content": "Calculate 48291.52 multiplied by 9182.11 using the multiplication tool."
    }
]

response = client.chat.completions.create(
    model="grok-4.7",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

assistant_message = response.choices[0].message

print("Grok's tool request:")
# print(assistant_message)
print(assistant_message.tool_calls)

Grok's tool request:
[ChatCompletionMessageFunctionToolCall(id='call-77ee3ad1-5b12-4bfb-8e7b-aaebfd4c0755-0', function=Function(arguments='{"a":48291.52,"b":9182.11}', name='multiply'), type='function')]


In [ ]:
import json

if assistant_message.tool_calls:

    tool_call = assistant_message.tool_calls[0]

    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    print("Function:", function_name)
    print("Arguments:", arguments)

    if function_name == "multiply":
        result = multiply(**arguments)
    else:
        result = {"error": "Unknown tool"}

    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps({"result": result})
    })

    final_response = client.chat.completions.create(
        model="grok-4.7",
        messages=messages
    )

    print("Final answer:")
    print(final_response.choices[0].message.content)

else:
    print("Grok answered directly:", assistant_message.content)


'''if — checks whether a tool call exists.
[0] — processes only the first tool call.
for — processes every tool call.
Return all tool results to Grok and generate the final answer.'''

    
# import json

# if assistant_message.tool_calls:
#     # Preserve Grok's original tool-call message
#     messages.append(assistant_message)

#     for tool_call in assistant_message.tool_calls:
#         function_name = tool_call.function.name
#         arguments = json.loads(tool_call.function.arguments)

#         print("Function:", function_name)
#         print("Arguments:", arguments)

#         # Execute the local Python function
#         if function_name == "multiply":
#             result = multiply(**arguments)
#         else:
#             result = {"error": "Unknown tool"}

#         # Send the result back to Grok
#         messages.append({
#             "role": "tool",
#             "tool_call_id": tool_call.id,
#             "content": json.dumps({"result": result})
#         })

#     # Let Grok explain the result to the user
#     final_response = client.chat.completions.create(
#         model="grok-4.7",
#         messages=messages,
#         tools=tools,
#         tool_choice="none"
#     )

#     print("\nFinal answer:")
#     print(final_response.choices[0].message.content)

# else:
#     print("Grok answered directly:", assistant_message.content)

Function: multiply
Arguments: {'a': 48291.52, 'b': 9182.11}
Python is executing: 48291.52 * 9182.11
Final answer:
**443,418,048.7072**


#2 tools

In [16]:
import os
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["XAI_API_KEY"],
    base_url="https://api.x.ai/v1"
)

MODEL = "grok-4.7"

Cell 2 — Define the two tools

In [17]:
def calculate(expression):
    # Demo only: fixed calculation for this example
    return {"result": 3 * 450 * 1.12}

def get_current_weather(city):
    # Mock weather data, not live weather
    weather_db = {
        "bangalore": {"temp_c": 24, "condition": "Pleasant Breeze"},
        "mumbai": {"temp_c": 31, "condition": "Humid and Sunny"}
    }
    return weather_db.get(city.lower(), {"error": "City not found"})

Cell 3 — Describe the tools to Grok

In [18]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calculate the total cost of 3 umbrellas at 450 INR each plus 12% tax.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get mock weather information for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    }
]

In [19]:
messages = [{
    "role": "user",
    "content": (
        "Tell me the weather in Bangalore and calculate "
        "the total cost of 3 umbrellas at 450 INR each plus 12% tax. "
     
    )
}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

assistant_message = response.choices[0].message
messages.append(assistant_message)

if assistant_message.tool_calls:
    print("Number of tool calls:", len(assistant_message.tool_calls))

    for tool_call in assistant_message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print("Executing:", name, args)

        if name == "calculate":
            result = calculate(**args)
        elif name == "get_current_weather":
            result = get_current_weather(**args)
        else:
            result = {"error": "Unknown tool"}

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="none"
    )

    print("\nFinal answer:")
    print(final_response.choices[0].message.content)

else:
    print(assistant_message.content)

Number of tool calls: 2
Executing: get_current_weather {'city': 'Bangalore'}
Executing: calculate {'expression': '3 * 450 * 1.12'}

Final answer:
**Weather in Bangalore:** 24°C, Pleasant Breeze.

**Umbrella cost:** 3 × 450 INR = 1,350 INR, plus 12% tax = **1,512 INR** total.


Program: Dependent tool calling with Grok


In [32]:
def search_product_price(product_name):
    products = {
        "wireless mouse": 1500,
        "mechanical keyboard": 4000
    }

    price = products.get(product_name.lower())

    if price is None:
        return {"error": "Product not found"}

    return {"product": product_name, "price": price}


def apply_discount(price, percent_off):
    discount = price * percent_off / 100
    return {
        "original_price": price,
        "discount": discount,
        "final_price": price - discount
    }

In [33]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_product_price",
            "description": "Find the price of a product in the catalog.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"}
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "apply_discount",
            "description": "Calculate the final price after a percentage discount using the given price.",
            "parameters": {
                "type": "object",
                "properties": {
                    "price": {"type": "number"},
                    "percent_off": {"type": "number"}
                },
                "required": ["price", "percent_off"]
            }
        }
    }
]

In [37]:
import json

messages = [
    {
        "role": "user",
        "content": "What is the price of a wireless mouse after a 20% discount?"
    }
]

for turn in range(5):

    response = client.chat.completions.create(
        model="grok-4.7",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    assistant_message = response.choices[0].message

    # No more tool calls: Grok has produced its final answer
    if not assistant_message.tool_calls:
        print("Final answer:", assistant_message.content)
        break

    messages.append(assistant_message)
    print(len(assistant_message.tool_calls), "tool calls requested by Grok.")
    # Execute every tool requested in this response
    for tool_call in assistant_message.tool_calls:
        

        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print(f"Executing {name}: {args}")

        if name == "search_product_price":
            result = search_product_price(**args)

        elif name == "apply_discount":
            result = apply_discount(**args)

        else:
            result = {"error": "Unknown tool"}

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

else:
    print("Stopped: maximum number of turns reached.")

1 tool calls requested by Grok.
Executing search_product_price: {'product_name': 'wireless mouse'}
1 tool calls requested by Grok.
Executing apply_discount: {'price': 1500.0, 'percent_off': 20.0}
Final answer: The wireless mouse costs **1200** after a 20% discount (original price 1500, discount amount 300).


Ollama

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # Placeholder; not a paid API key
)

MODEL = "llama3.2:latest"

In [39]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain function calling in two simple sentences."
        }
    ]
)

print(response.choices[0].message.content)

A function in programming is a block of code that performs a specific task, and when called, it executes the code within that block to achieve the desired result. By calling a function, you're essentially instructing the program to run the specified code, allowing you to reuse and organize your code more efficiently.
